In [ ]:
# 3D 데이터 진단
print("=== 3D 데이터 진단 ===")
print(f"X shape: {X.shape}")
print(f"X dtype: {X.dtype}")
print(f"전체 min/max: {X.min():.2f} / {X.max():.2f}")

# 각 좌표별 범위 확인
x_vals = X[:, :, 0::3]  # x좌표들
y_vals = X[:, :, 1::3]  # y좌표들
z_vals = X[:, :, 2::3]  # z좌표들

print(f"\nx좌표 범위: {x_vals.min():.2f} ~ {x_vals.max():.2f}, std: {x_vals.std():.2f}")
print(f"y좌표 범위: {y_vals.min():.2f} ~ {y_vals.max():.2f}, std: {y_vals.std():.2f}")
print(f"z좌표 범위: {z_vals.min():.2f} ~ {z_vals.max():.2f}, std: {z_vals.std():.2f}")

# 0값 비율 (빈 키포인트 확인)
zero_ratio = (X == 0).sum() / X.size * 100
print(f"\n0값 비율: {zero_ratio:.1f}%")

# 샘플 간 분산 확인 (다양성)
sample_means = X.mean(axis=(1,2))
print(f"\n샘플 평균의 분산: {sample_means.std():.2f}")

# 같은 클래스끼리 유사도 vs 다른 클래스
from numpy.random import choice
same_dists = []
diff_dists = []
for _ in range(1000):
    i, j = choice(len(X), 2, replace=False)
    dist = np.linalg.norm(X[i].flatten() - X[j].flatten())
    if y[i] == y[j]:
        same_dists.append(dist)
    else:
        diff_dists.append(dist)

print(f"\n같은 클래스 거리 평균: {np.mean(same_dists):.1f}")
print(f"다른 클래스 거리 평균: {np.mean(diff_dists):.1f}")
print(f"비율 (같은/다른): {np.mean(same_dists)/np.mean(diff_dists):.3f}")
print(f"→ 1.0에 가까우면 클래스 구분이 안 되는 데이터!")

=== 3D 데이터 진단 ===
X shape: (15000, 30, 411)
X dtype: float32
전체 min/max: -753.42 / 2540.08

x좌표 범위: -753.42 ~ 449.99, std: 43.98
y좌표 범위: -519.30 ~ 527.17, std: 120.31
z좌표 범위: 1.44 ~ 2540.08, std: 747.53

0값 비율: 0.0%

샘플 평균의 분산: 243.46

같은 클래스 거리 평균: nan
다른 클래스 거리 평균: 33931.3
비율 (같은/다른): nan
→ 1.0에 가까우면 클래스 구분이 안 되는 데이터!


In [ ]:
# z좌표 분포 상세 확인
print("=== z좌표 상세 분석 ===")
z_vals = X[:, :, 2::3]
print(f"z 중앙값: {np.median(z_vals):.1f}")
print(f"z 평균: {z_vals.mean():.1f}")
print(f"z 25%: {np.percentile(z_vals, 25):.1f}")
print(f"z 75%: {np.percentile(z_vals, 75):.1f}")

# z값 분포 히스토그램 (텍스트)
hist, edges = np.histogram(z_vals.flatten(), bins=10)
print(f"\nz값 분포:")
for i in range(len(hist)):
    bar = '■' * (hist[i] // (hist.max() // 30 + 1))
    print(f"  {edges[i]:7.0f}~{edges[i+1]:7.0f}: {hist[i]:>10,} {bar}")

# 2D 데이터와 3D 데이터의 x,y 비교
# 3D의 x,y만 추출해서 2D처럼 사용하면?
X_xy_only = X[:, :, [i for i in range(411) if i % 3 != 2]]  # z 제거
print(f"\nx,y만 추출: {X_xy_only.shape}")
print(f"x,y std: {X_xy_only.std():.2f}")

# 클래스 내 분산 vs 클래스 간 분산 (일부만)
classes_to_check = np.unique(y)[:20]
within_var = []
for c in classes_to_check:
    mask = y == c
    if mask.sum() >= 2:
        class_data = X[mask].reshape(mask.sum(), -1)
        within_var.append(class_data.var(axis=0).mean())

print(f"\n클래스 내 평균 분산: {np.mean(within_var):.1f}")
print(f"전체 분산: {X.reshape(len(X), -1).var(axis=0).mean():.1f}")
print(f"비율 (내/전체): {np.mean(within_var)/X.reshape(len(X),-1).var(axis=0).mean():.3f}")
print(f"-> 1.0에 가까우면 클래스별 차이가 거의 없는 것")

=== z좌표 상세 분석 ===
z 중앙값: 2.4
z 평균: 280.7
z 25%: 2.3
z 75%: 2.4

z값 분포:
        1~    255: 54,128,700 █████████████████████████████
      255~    509:          0 
      509~    763:          0 
      763~   1017:          0 
     1017~   1271:          0 
     1271~   1525:     17,250 
     1525~   1778:     17,255 
     1778~   2032:     97,240 
     2032~   2286:  3,021,610 █
     2286~   2540:  4,367,945 ██

x,y만 추출: (15000, 30, 274)
x,y std: 90.59

클래스 내 평균 분산: 0.0
전체 분산: 191098.2
비율 (내/전체): 0.000
→ 1.0에 가까우면 클래스별 차이가 거의 없는 것


In [ ]:
# 키포인트 그룹별 z값 분석
# 왼손63(0~62) + 오른손63(63~125) + 얼굴210(126~335) + 상체75(336~410)

z_left  = X[:, :, 2:63:3]       # 왼손 z
z_right = X[:, :, 65:126:3]     # 오른손 z
z_face  = X[:, :, 128:336:3]    # 얼굴 z
z_pose  = X[:, :, 338:411:3]    # 상체 z

print("=== 그룹별 z값 범위 ===")
print(f"왼손 z:  min={z_left.min():.1f}, max={z_left.max():.1f}, median={np.median(z_left):.1f}, std={z_left.std():.1f}")
print(f"오른손 z: min={z_right.min():.1f}, max={z_right.max():.1f}, median={np.median(z_right):.1f}, std={z_right.std():.1f}")
print(f"얼굴 z:  min={z_face.min():.1f}, max={z_face.max():.1f}, median={np.median(z_face):.1f}, std={z_face.std():.1f}")
print(f"상체 z:  min={z_pose.min():.1f}, max={z_pose.max():.1f}, median={np.median(z_pose):.1f}, std={z_pose.std():.1f}")

=== 그룹별 z값 범위 ===
왼손 z:  min=1.8, max=2411.0, median=2.4, std=739.9
오른손 z: min=1.5, max=2507.3, median=2.3, std=735.6
얼굴 z:  min=2.1, max=2540.1, median=2.3, std=748.6
상체 z:  min=1.4, max=2517.1, median=2.4, std=760.7


In [ ]:
# 각도별 z값 확인 (각 단어당 5각도 = 5샘플)
print("=== 각도별 z값 분석 ===")
print("(각 단어 × 5각도 -> D,F,L,R,U 순서로 저장되었을 가능성)")

# 첫 10개 단어의 z 중앙값 확인 (각 단어당 5샘플)
for word_id in range(5):
    start = word_id * 5
    end = start + 5
    if end > len(X):
        break
    for i, angle in enumerate(['D', 'F', 'L', 'R', 'U']):
        idx = start + i
        z_med = np.median(X[idx, :, 2::3])
        z_max = X[idx, :, 2::3].max()
        print(f"  WORD{word_id+1:04d}_{angle}: z 중앙값={z_med:.1f}, z 최대={z_max:.1f}")
    print()

=== 각도별 z값 분석 ===
(각 단어 × 5각도 → D,F,L,R,U 순서로 저장되었을 가능성)
  WORD0001_D: z 중앙값=2.3, z 최대=2.5
  WORD0001_F: z 중앙값=2.3, z 최대=2.5
  WORD0001_L: z 중앙값=2.3, z 최대=2.5
  WORD0001_R: z 중앙값=2.3, z 최대=2.5
  WORD0001_U: z 중앙값=2.3, z 최대=2.5

  WORD0002_D: z 중앙값=2.3, z 최대=2.5
  WORD0002_F: z 중앙값=2.3, z 최대=2.5
  WORD0002_L: z 중앙값=2.3, z 최대=2.5
  WORD0002_R: z 중앙값=2.3, z 최대=2.5
  WORD0002_U: z 중앙값=2.3, z 최대=2.5

  WORD0003_D: z 중앙값=2.3, z 최대=2.4
  WORD0003_F: z 중앙값=2.3, z 최대=2.4
  WORD0003_L: z 중앙값=2.3, z 최대=2.4
  WORD0003_R: z 중앙값=2.3, z 최대=2.4
  WORD0003_U: z 중앙값=2.3, z 최대=2.4

  WORD0004_D: z 중앙값=2.3, z 최대=2.4
  WORD0004_F: z 중앙값=2.3, z 최대=2.4
  WORD0004_L: z 중앙값=2.3, z 최대=2.4
  WORD0004_R: z 중앙값=2.3, z 최대=2.4
  WORD0004_U: z 중앙값=2.3, z 최대=2.4

  WORD0005_D: z 중앙값=2.3, z 최대=2.5
  WORD0005_F: z 중앙값=2.3, z 최대=2.5
  WORD0005_L: z 중앙값=2.3, z 최대=2.5
  WORD0005_R: z 중앙값=2.3, z 최대=2.5
  WORD0005_U: z 중앙값=2.3, z 최대=2.5



In [ ]:
# z값이 큰 샘플 찾기
z_per_sample = np.median(X[:, :, 2::3], axis=(1, 2))  # 각 샘플의 z 중앙값

# z가 100 이상인 샘플
big_z_mask = z_per_sample > 100
print(f"z 중앙값 > 100인 샘플: {big_z_mask.sum()}개 / {len(X)}개 ({big_z_mask.sum()/len(X)*100:.1f}%)")

# 해당 단어 ID 확인
big_z_words = np.unique(y[big_z_mask])
print(f"해당 단어 수: {len(big_z_words)}개")
print(f"단어 ID 예시: {big_z_words[:20]}")

# z 중앙값 분포 요약
print(f"\n=== 샘플별 z 중앙값 분포 ===")
print(f"min: {z_per_sample.min():.1f}")
print(f"25%: {np.percentile(z_per_sample, 25):.1f}")
print(f"50%: {np.percentile(z_per_sample, 50):.1f}")
print(f"75%: {np.percentile(z_per_sample, 75):.1f}")
print(f"90%: {np.percentile(z_per_sample, 90):.1f}")
print(f"95%: {np.percentile(z_per_sample, 95):.1f}")
print(f"max: {z_per_sample.max():.1f}")

z 중앙값 > 100인 샘플: 1830개 / 15000개 (12.2%)
해당 단어 수: 366개
단어 ID 예시: [1100 1101 1102 1103 1104 1105 1106 1107 1108 1109 1110 1111 1112 1113
 1114 1115 1116 1117 1118 1119]

=== 샘플별 z 중앙값 분포 ===
min: 2.3
25%: 2.3
50%: 2.3
75%: 2.4
90%: 2277.5
95%: 2295.2
max: 2323.4


In [ ]:
# 2D 데이터와 비교
download_from_gcs(BUCKET_NAME, 'processed/X_train.npy', '/content/X_train_2d.npy')
X_2d = np.load('/content/X_train_2d.npy')

xy_indices = [i for i in range(411) if i % 3 != 2]
X_3d_xy = X[:, :, xy_indices]

print(f'2D shape: {X_2d.shape}')
print(f'3D에서 z제거: {X_3d_xy.shape}')

if X_2d.shape == X_3d_xy.shape:
    is_same = np.allclose(X_2d, X_3d_xy, atol=0.01)
    diff = np.abs(X_2d - X_3d_xy).max()
    print(f'동일한 데이터인가: {is_same}')
    print(f'최대 차이: {diff:.4f}')
else:
    print('shape이 달라서 직접 비교 불가')

Downloaded: processed/X_train.npy → /content/X_train_2d.npy
2D shape: (15000, 30, 274)
3D에서 z제거: (15000, 30, 274)
동일한 데이터인가: False
최대 차이: 2075.8999


In [ ]:
# 3D x,y와 2D 데이터 분포 비교
X_2d = np.load('/content/X_train_2d.npy')

xy_indices = [i for i in range(411) if i % 3 != 2]
X_3d_xy = X[:, :, xy_indices]

print("=== 2D 데이터 ===")
print(f"min: {X_2d.min():.1f}, max: {X_2d.max():.1f}")
print(f"mean: {X_2d.mean():.1f}, std: {X_2d.std():.1f}")
print(f"0값 비율: {(X_2d == 0).sum() / X_2d.size * 100:.1f}%")

print(f"\n=== 3D x,y 데이터 ===")
print(f"min: {X_3d_xy.min():.1f}, max: {X_3d_xy.max():.1f}")
print(f"mean: {X_3d_xy.mean():.1f}, std: {X_3d_xy.std():.1f}")
print(f"0값 비율: {(X_3d_xy == 0).sum() / X_3d_xy.size * 100:.1f}%")

# 정규화 후 비교
X_2d_norm = (X_2d - X_2d.mean(axis=(0,1), keepdims=True)) / (X_2d.std(axis=(0,1), keepdims=True) + 1e-8)
X_3d_norm = (X_3d_xy - X_3d_xy.mean(axis=(0,1), keepdims=True)) / (X_3d_xy.std(axis=(0,1), keepdims=True) + 1e-8)

print(f"\n=== 정규화 후 ===")
print(f"2D norm - min: {X_2d_norm.min():.2f}, max: {X_2d_norm.max():.2f}, std: {X_2d_norm.std():.2f}")
print(f"3D norm - min: {X_3d_norm.min():.2f}, max: {X_3d_norm.max():.2f}, std: {X_3d_norm.std():.2f}")

# 클래스별 분산 비교 (학습 가능성 지표)
within_2d = []
within_3d = []
for c in np.unique(y)[:100]:
    mask = y == c
    if mask.sum() >= 2:
        d2 = X_2d_norm[mask].reshape(mask.sum(), -1)
        d3 = X_3d_norm[mask].reshape(mask.sum(), -1)
        within_2d.append(d2.var(axis=0).mean())
        within_3d.append(d3.var(axis=0).mean())

total_2d = X_2d_norm.reshape(len(X_2d_norm), -1).var(axis=0).mean()
total_3d = X_3d_norm.reshape(len(X_3d_norm), -1).var(axis=0).mean()

print(f"\n=== 클래스 내 분산 / 전체 분산 (낮을수록 클래스 구분 좋음) ===")
print(f"2D: {np.mean(within_2d):.4f} / {total_2d:.4f} = {np.mean(within_2d)/total_2d:.4f}")
print(f"3D: {np.mean(within_3d):.4f} / {total_3d:.4f} = {np.mean(within_3d)/total_3d:.4f}")

=== 2D 데이터 ===
min: -403.6, max: 2076.3
mean: 691.7, std: 326.2
0값 비율: 0.6%

=== 3D x,y 데이터 ===
min: -753.4, max: 527.2
mean: -3.2, std: 90.6
0값 비율: 0.0%

=== 정규화 후 ===
2D norm - min: -23.94, max: 11.92, std: 1.00
3D norm - min: -24.39, max: 14.36, std: 1.00

=== 클래스 내 분산 / 전체 분산 (낮을수록 클래스 구분 좋음) ===
2D: 0.4381 / 0.8957 = 0.4891
3D: 0.0000 / 0.9929 = 0.0000


In [ ]:
# 같은 단어의 5개 샘플이 동일한지 확인
xy_indices = [i for i in range(411) if i % 3 != 2]
X_3d_xy = X[:, :, xy_indices]

# WORD0001의 5개 샘플 비교
for word_id in [0, 1, 2, 100, 500]:
    mask = y == word_id
    samples = X_3d_xy[mask]
    if len(samples) >= 2:
        # 샘플 간 차이
        diffs = []
        for i in range(len(samples)):
            for j in range(i+1, len(samples)):
                diffs.append(np.abs(samples[i] - samples[j]).max())
        print(f"WORD{word_id+1:04d}: {len(samples)}개 샘플, 최대 차이={max(diffs):.6f}")

WORD0001: 5개 샘플, 최대 차이=0.000000
WORD0002: 5개 샘플, 최대 차이=0.000000
WORD0003: 5개 샘플, 최대 차이=0.000000
WORD0101: 5개 샘플, 최대 차이=0.000000
WORD0501: 5개 샘플, 최대 차이=0.000000


In [ ]:
# 셀 1 — GCS 인증
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'project-id'
BUCKET_NAME = 'sign-language-data-2026'

!gcloud config set project {PROJECT_ID}

Project 'project-id' lacks an 'environment' tag. Please create or add a tag with key 'environment' and a value like 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`. See https://cloud.google.com/resource-manager/docs/creating-managing-projects#designate_project_environments_with_tags for details.
Updated property [core/project].


In [ ]:
# 셀 2 — 데이터 로드
import numpy as np
from google.cloud import storage

def download_from_gcs(bucket_name, blob_name, local_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.download_to_filename(local_path)
    print(f'Downloaded: {blob_name} → {local_path}')

download_from_gcs(BUCKET_NAME, 'processed/X_train_3d.npy', '/content/X_train_3d.npy')
download_from_gcs(BUCKET_NAME, 'processed/y_train_3d.npy', '/content/y_train_3d.npy')

X = np.load('/content/X_train_3d.npy')
y = np.load('/content/y_train_3d.npy')

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'클래스 수: {len(np.unique(y))}')

Downloaded: processed/X_train_3d.npy → /content/X_train_3d.npy
Downloaded: processed/y_train_3d.npy → /content/y_train_3d.npy
X shape: (15000, 30, 411)
y shape: (15000,)
클래스 수: 3000


In [ ]:
# 셀 3 — 전처리
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import numpy as np

# z좌표 제거 -> x,y만 사용 (z는 88%가 상수 2.3으로 정보 없음)
xy_indices = [i for i in range(411) if i % 3 != 2]
X_xy = X[:, :, xy_indices]  # (15000, 30, 274)
print(f'z 제거 후: {X_xy.shape}')

# Z-score 정규화 
X_mean = X_xy.mean(axis=(0, 1), keepdims=True)
X_std  = X_xy.std(axis=(0, 1), keepdims=True) + 1e-8
X_norm = (X_xy - X_mean) / X_std

X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

NUM_CLASSES = 3000
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

z 제거 후: (15000, 30, 274)
Train: (10500, 30, 274), Val: (2250, 30, 274), Test: (2250, 30, 274)


In [ ]:
# 셀 4 - 증강(3배)
import gc

def augment(x):
    noise = np.random.normal(0, 0.01, x.shape)
    scale = np.random.uniform(0.9, 1.1)
    shift = np.random.uniform(-0.05, 0.05)
    return (x * scale + noise + shift).astype(np.float32)

X_train = X_train.astype(np.float32)

print("증강 1/2...")
X_aug1 = np.array([augment(x) for x in X_train], dtype=np.float32)
print("증강 2/2...")
X_aug2 = np.array([augment(x) for x in X_train], dtype=np.float32)

X_train_final = np.concatenate([X_train, X_aug1, X_aug2], axis=0)
del X_aug1, X_aug2
gc.collect()

y_train_final = np.concatenate([y_train, y_train, y_train], axis=0)
y_train_final_cat = to_categorical(y_train_final, NUM_CLASSES)

del X_norm
gc.collect() # 메로리 해제

print(f'원본: {X_train.shape}')
print(f'증강 후: {X_train_final.shape}')

증강 1/2...
증강 2/2...
원본: (10500, 30, 274)
증강 후: (31500, 30, 274)


In [ ]:
# 셀 5 — 모델 정의 (transformer)
import tensorflow as tf
from tensorflow.keras import layers, models

print(f'GPU: {tf.config.list_physical_devices("GPU")}')

def positional_encoding(length, depth):
    positions = tf.range(length, dtype=tf.float32)[:, tf.newaxis]
    dims = tf.range(depth, dtype=tf.float32)[tf.newaxis, :]
    angles = positions / tf.pow(10000.0, (2 * (dims // 2)) / tf.cast(depth, tf.float32))
    angles = tf.concat([tf.sin(angles[:, 0::2]), tf.cos(angles[:, 1::2])], axis=-1)
    return angles[tf.newaxis, :, :]

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim)
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x)
        x = self.norm1(x + self.drop1(attn, training=training))
        ffn = self.ffn(x)
        x = self.norm2(x + self.drop2(ffn, training=training))
        return x

def build_transformer_model(input_shape=(30, 274), num_classes=3000,
                             embed_dim=128, num_heads=4, ff_dim=256,
                             num_blocks=3, dropout=0.1):
    inputs = layers.Input(shape=input_shape)
    x = layers.Dense(embed_dim)(inputs)
    pos_enc = positional_encoding(input_shape[0], embed_dim)
    x = x + pos_enc
    for _ in range(num_blocks):
        x = TransformerBlock(embed_dim, num_heads, ff_dim, dropout)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

model = build_transformer_model()
model.summary()

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 30, 274)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 30, 128)        │        35,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_1 (Add)                     │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 30, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, 30, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ (None, 30, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 3000)           │       771,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,236,664 (4.72 MB)

 Trainable params: 1,236,664 (4.72 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 셀 6 — 학습
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('/content/best_3d_model.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

history = model.fit(
    X_train_final, y_train_final_cat,
    validation_data=(X_val, y_val_cat),
    epochs=100,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/100
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 6.8561e-04 - loss: 7.8859 - top5_acc: 0.0031
Epoch 1: val_accuracy improved from None to 0.00044, saving model to /content/best_3d_model.h5



Epoch 1: finished saving model to /content/best_3d_model.h5
493/493 ━━━━━━━━━━━━━━━━━━━━ 36s 35ms/step - accuracy: 5.3968e-04 - loss: 7.7984 - top5_acc: 0.0027 - val_accuracy: 4.4444e-04 - val_loss: 7.8151 - val_top5_acc: 4.4444e-04 - learning_rate: 0.0010
Epoch 2/100
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 4.8183e-04 - loss: 7.6440 - top5_acc: 0.0043
Epoch 2: val_accuracy did not improve from 0.00044
493/493 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 4.4444e-04 - loss: 7.6501 - top5_acc: 0.0031 - val_accuracy: 4.4444e-04 - val_loss: 7.8508 - val_top5_acc: 0.0013 - learning_rate: 0.0010
Epoch 3/100
491/493 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0011 - loss: 7.6249 - top5_acc: 0.0042
Epoch 3: val_accuracy did not improve from 0.00044
493/493 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 7.9365e-04 - loss: 7.6312 - top5_acc: 0.0030 - val_accuracy: 0.0000e+00 - val_loss: 7.8475 - val_top5_acc: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/100
491/493 ━━━━━━━━━━━━━━━

In [ ]:
# 셀 7 — 평가 및 시각화
import matplotlib.pyplot as plt

test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'=============================')
print(f'  Test Top-1 Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'  Test Top-5 Accuracy: {test_top5:.4f} ({test_top5*100:.1f}%)')
print(f'  Test Loss:           {test_loss:.4f}')
print(f'=============================')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Top-1 Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['top5_acc'], label='Train')
axes[1].plot(history.history['val_top5_acc'], label='Val')
axes[1].set_title('Top-5 Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(history.history['loss'], label='Train')
axes[2].plot(history.history['val_loss'], label='Val')
axes[2].set_title('Loss'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curve_3d.png', dpi=150)
plt.show()

In [ ]:
# 셀 8 — GCS 업로드
def upload_to_gcs(local_path, bucket_name, blob_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)
    print(f'Uploaded: {local_path} → gs://{bucket_name}/{blob_name}')

# 정규화 파라미터 저장
np.save('/content/X_mean_3d.npy', X_mean)
np.save('/content/X_std_3d.npy',  X_std)

upload_to_gcs('/content/best_3d_model.h5',       BUCKET_NAME, 'models/transformer_3d.h5')
upload_to_gcs('/content/X_mean_3d.npy',           BUCKET_NAME, 'models/X_mean_3d.npy')
upload_to_gcs('/content/X_std_3d.npy',            BUCKET_NAME, 'models/X_std_3d.npy')
upload_to_gcs('/content/training_curve_3d.png',   BUCKET_NAME, 'models/training_curve_3d.png')

print('3D 모델 + 정규화 파라미터 GCS 업로드 완료')

In [ ]:
# 셀 9 — 추론 테스트
import json

download_from_gcs(BUCKET_NAME, 'processed/label_map.json', '/content/label_map.json')
with open('/content/label_map.json', 'r', encoding='utf-8') as f:
    label_map = json.load(f)

def predict_word_3d(raw_keypoints_30x411, top_k=5):
    """원본 3D 키포인트(30,411) -> z제거 -> 정규화 -> 예측"""
    xy = raw_keypoints_30x411[:, xy_indices]  # z 제거
    x = (xy - X_mean[0]) / X_std[0]
    x = x[np.newaxis, ...]
    probs = model.predict(x, verbose=0)[0]
    top_indices = probs.argsort()[-top_k:][::-1]
    return [(label_map[str(i + 1)], float(probs[i])) for i in top_indices]

# 테스트
sample_norm = X_test[0]
sample_raw = sample_norm * X_std[0] + X_mean[0]  # 역정규화

# predict_word_3d는 원본 411차원 입력을 기대하므로, 274차원으로 직접 테스트
x_input = sample_norm[np.newaxis, ...]
probs = model.predict(x_input, verbose=0)[0]
top_indices = probs.argsort()[-5:][::-1]

true_label = label_map[str(int(y_test[0]) + 1)]
print(f'정답: {true_label}')
print(f'예측 Top-5:')
for i in top_indices:
    word = label_map[str(i + 1)]
    mark = '○' if word == true_label else '  '
    print(f'  {mark} {word}: {probs[i]:.4f}')